In [1]:
from transformers import BitsAndBytesConfig
import torch

bnb_4bit_compute_dtype = torch.float32

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=bnb_4bit_compute_dtype,
)

In [2]:
from datasets import load_dataset

dataset = load_dataset("neo4j/text2cypher-2024v1", download_mode="force_redownload")


def format_instruction(sample):
    return {
        "text": f"""<|im_start|>system
        You are a Cypher query expert. Generate valid Cypher queries based on the input question and database schema.
        Schema: {sample["schema"]}<|im_end|>
        <|im_start|>user
        {sample["question"]}<|im_end|>
        <|im_start|>assistant
        {sample["cypher"]}"""
    }


dataset["train"] = dataset["train"].map(format_instruction)
dataset["test"] = dataset["test"].map(format_instruction)

train-00000-of-00001.parquet:   0%|          | 0.00/7.33M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/835k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/39554 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/4833 [00:00<?, ? examples/s]

Map:   0%|          | 0/39554 [00:00<?, ? examples/s]

Map:   0%|          | 0/4833 [00:00<?, ? examples/s]

In [ ]:
from transformers import (
    AutoModelForCausalLM,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    Trainer,
)
from unsloth import FastLanguageModel  # Optimized training
import huggingface_hub

huggingface_hub.constants.HF_HUB_ENABLE_HF_TRANSFER = False
huggingface_hub.constants.DEFAULT_ETAG_TIMEOUT = 30  # Increase this value if needed

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-Coder-0.5B-Instruct",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
    quantization_config=quant_config,
    # compute_dtype=bnb_4bit_compute_dtype
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.2.15: Fast Qwen2 patching. Transformers: 4.49.0.
   \\   /|    GPU: NVIDIA GeForce RTX 3090. Max memory: 23.582 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.6. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512,
        return_tensors="pt",
    )


tokenized_datasets = dataset.map(
    tokenize_function, batched=True, remove_columns=dataset["train"].column_names
)

Map:   0%|          | 0/39554 [00:00<?, ? examples/s]

Map:   0%|          | 0/4833 [00:00<?, ? examples/s]

In [5]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,  # Rank decomposition dimension
    lora_alpha=32,  # Scaling factor
    target_modules=["q_proj", "v_proj"],  # Attention submodules
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    modules_to_save=["lm_head"],  # Maintain output layer flexibility
)

model = FastLanguageModel.get_peft_model(
    model,
    r=lora_config.r,  # or lora_config['r'] if it's a dictionary
    target_modules=lora_config.target_modules,
    lora_alpha=lora_config.lora_alpha,
    lora_dropout=lora_config.lora_dropout,
    bias=lora_config.bias,
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

training_args = TrainingArguments(
    output_dir="./text2cypher-lora",
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    num_train_epochs=3,
    logging_steps=50,
    evaluation_strategy="steps",
    eval_steps=100,
    optim="adamw_torch",
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    report_to="none",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2025.2.15 patched 24 layers with 0 QKV layers, 0 O layers and 0 MLP layers.
/home/TomKerby/School/USU-LLM-Class/.venv/lib/python3.12/site-packages/transformers/training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [6]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 39,554 | Num Epochs = 3
O^O/ \_/ \    Batch size per device = 16 | Gradient Accumulation steps = 2
\        /    Total batch size = 32 | Total steps = 3,708
 "-____-"     Number of trainable parameters = 1,081,344


Step,Training Loss,Validation Loss
100,2.367000,2.142753
200,2.127000,1.969911
300,1.877300,1.754945
400,1.515200,1.502223
500,1.306500,1.377811
600,1.182900,1.232378
700,1.020800,1.079471
800,0.832300,0.940914
900,0.676000,0.821384
1000,0.611600,0.738022


Unsloth: Not an error, but Qwen2ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


TrainOutput(global_step=3708, training_loss=0.6038893851518888, metrics={'train_runtime': 7285.3891, 'train_samples_per_second': 16.288, 'train_steps_per_second': 0.509, 'total_flos': 1.3078587667591987e+17, 'train_loss': 0.6038893851518888, 'epoch': 2.997978164173069})

In [7]:
model.save_pretrained("./text2cypher-lora-adapter")
tokenizer.save_pretrained("./text2cypher-lora-adapter")

('./text2cypher-lora-adapter/tokenizer_config.json',
 './text2cypher-lora-adapter/special_tokens_map.json',
 './text2cypher-lora-adapter/vocab.json',
 './text2cypher-lora-adapter/merges.txt',
 './text2cypher-lora-adapter/added_tokens.json',
 './text2cypher-lora-adapter/tokenizer.json')

In [11]:
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-Coder-0.5B-Instruct")
merged_model = PeftModel.from_pretrained(base_model, "./text2cypher-lora-adapter")
merged_model = merged_model.merge_and_unload()